#### **02-search**

The same crossover rule, with its two lengths searched instead of chosen.

**A search returns two numbers and only one of them is worth quoting.** The
first is the best score it found on the bars it was allowed to see, and it is
inflated by the act of searching, by more the harder you search. The second is
that same setting scored on bars no trial ever touched. The gap between them is
what the search invented.

The last section stops trusting one split and refits along the
series instead.

#### **Setup**

The environment is built before this cell runs. `docker compose up` installs
both libraries and starts a router. A pip install next to a router you started
yourself does the same thing.

So this cell checks and opens the client. If either library is missing, or
nothing answers on the router address, it stops here and prints the command to
run.

Searching needs one thing beyond a backtest. `tune` drives
[optuna](https://optuna.org) and ships trials to worker processes with
cloudpickle, both of which arrive with the `tune` extra rather than with
[emsl](https://github.com/atOCEANO/embeddable-market-simulation-library)
itself.

In [1]:
import os
import urllib.request

ROUTER_URL = os.environ.get(key="ROUTER_URL", default="http://127.0.0.1:8040")

try:
    import emsl
    import exchange_router_client
    import optuna
except ImportError as error:
    raise ImportError(f"{error.name} is missing; run 'docker compose up' from "
                      f"this repository, or pip install -r requirements.txt") from None


def router_is_up(url):
    try:
        with urllib.request.urlopen(url=f"{url}/status", timeout=2.0) as response:
            return response.status == 200
    except Exception:
        return False


if not router_is_up(url=ROUTER_URL):
    raise RuntimeError(f"no router answering on {ROUTER_URL}; run 'docker compose "
                       f"up' from this repository, or start one yourself and point "
                       f"ROUTER_URL at it")

optuna.logging.set_verbosity(verbosity=optuna.logging.WARNING)

client = exchange_router_client.ExchangeRouterClient(base_url=ROUTER_URL)

print("service    ", client.get_version(), "at", ROUTER_URL)
print("emsl       ", emsl.__version__)
print("optuna     ", optuna.__version__)

service     2.5.6 at http://exchange-router-service:8040
emsl        1.3.1
optuna      5.0.0


#### **The candles**

One year of hourly bars, pinned so the run is over the same bars every time.
**`start` is the end of the window and the router walks backwards from it**, so
this asks for the year that ends on 1 January 2026, which is 2025.

A search makes the pinning matter more than a single backtest does. A moving
window would change the winner as well as its score, so two runs of this
notebook would disagree about the answer and not only about the number.

A frame carries metadata, not just columns. `candles.attrs` holds what is
constant across the request: the venue, the symbol, the quote asset, the unit
volume is counted in, and the schema number. The same idea returns further
down: every run is stamped with a hash of the bars it saw, and that hash is
how two rows of a comparison prove they are not the same test twice.

In [2]:
import datetime

EXCHANGE = "binance"
MARKET   = "spot"
SYMBOL   = "BTCUSDT"
INTERVAL = "1h"

ANCHOR = int(datetime.datetime(year=2026, month=1, day=1,
                               tzinfo=datetime.timezone.utc).timestamp() * 1000)
BARS   = 8760

candles = client.get_candles(
    exchange=EXCHANGE,
    market_type=MARKET,
    symbol=SYMBOL,
    interval=INTERVAL,
    limit=BARS,
    start=ANCHOR,
)

print(f"{len(candles)} bars, {candles.index[0]} to {candles.index[-1]}")
print(f"columns  {', '.join(candles.columns)}")

for key in ("exchange", "market_type", "symbol", "quote", "volume_unit",
            "schema_version"):
    print(f"  {key:<16} {candles.attrs[key]}")

8760 bars, 2025-01-01 01:00:00+00:00 to 2026-01-01 00:00:00+00:00
columns  open, high, low, close, volume, volume_usd
  exchange         binance
  market_type      spot
  symbol           BTCUSDT
  quote            USDT
  volume_unit      base
  schema_version   3


#### **The rule**

The same crossover, unchanged. **What matters for a search is that the
tunables are constructor arguments**, because each trial builds a fresh
strategy by calling `EmaCross(**params)` with the values it sampled. That is
`__init__`; the engine then calls `init` on the object it produced.

`warmup` is computed in `init` from `slow`, which is why it has to be
settable there rather than declared on the class. A search that samples
`slow` changes the warm-up on every trial.

A class is handed to the search rather than an instance, because an
instance would be one point in the space rather than the shape of it.

`marks()` earns its place here rather than in a single backtest. The search
picks the windows, so a legend typed by hand names whatever you had in mind
when you wrote the cell and not what won. Reading `self.fast_length` instead
makes the charts below correct without anyone remembering to update them.
emsl never calls the method, so the trials do not pay for it.

In [3]:
class EmaCross(emsl.Strategy):

    def __init__(self, fast, slow, weight=0.95):    #  the tunables
        self.fast_length = fast
        self.slow_length = slow
        self.weight      = weight

    def init(self, engine):    #  emsl's hook, once per run
        self.fast = emsl.ta.ema(values=engine.closes, length=self.fast_length)
        self.slow = emsl.ta.ema(values=engine.closes, length=self.slow_length)

        self.up   = emsl.ta.crossover(values=self.fast, other=self.slow)
        self.down = emsl.ta.crossunder(values=self.fast, other=self.slow)

        self.warmup = self.slow_length

    def next(self, state, engine):
        i = state["tick_index"]

        if state["position"] == 0.0 and self.up[i]:
            engine.market_buy(size=engine.qty_from_weight(fraction=self.weight))

        elif state["position"] > 0.0 and self.down[i]:
            engine.close()

    def marks(self):
        return [
            emsl.plot.Line(values=self.fast, name=f"EMA {self.fast_length}"),
            emsl.plot.Line(values=self.slow, name=f"EMA {self.slow_length}",
                           style="dashed"),
        ]

#### **The search**

Every trial is a full backtest through the same `Backtester`, at the same
venue, with the same fill model. **The strategy you tune is the strategy you
backtested**, which is why the venue is one object rather than a set of
keyword arguments repeated at three call sites.

**`oos=0.3` is the line that makes the result mean anything.** Every trial is
fitted on the first seventy percent of the series, and the winner alone is then
scored on the last thirty, which no trial ever saw. Leave it out and there is no
second number, only the flattering one.

`n_jobs=1` is a deliberate choice and it costs wall clock. A parallel
search asks for several trials before any has reported, so the order results
reach the sampler depends on which worker finishes first, the sampler sees a
different history, and it suggests different points. Nothing is broken; that
is what asynchronous search is. But a search that does not reproduce costs an
afternoon to chase, so pin it to one when a result has to reproduce and use
`-1` when you are exploring.

`objective` takes any key from the stats dict, or a function of your own over
the finished `BacktestResult`, which puts the equity curve and every trade in
reach. `min_trades` discards configurations that barely traded, which otherwise
win by not participating.

In [4]:
VENUE = emsl.Market(
    kind="spot",
    quote=10_000.0,
    fee_taker=0.0006,
    fee_maker=0.0002,
    slippage_bps=2.0,
)

SPACE = {"fast": (5, 40), "slow": (40, 200)}

study = VENUE.tune(
    strategy=EmaCross,
    space=SPACE,
    data=candles,
    objective="sharpe",
    n_trials=200,
    oos=0.3,
    n_jobs=1,
    seed=7,
    min_trades=10,
)

print(f"searched {len(study.trials)} trials over {SPACE}")
print(f"best      {study.best_params}")

searched 200 trials over {'fast': (5, 40), 'slow': (40, 200)}
best      {'fast': 19, 'slow': 197}


#### **The two numbers**

**`best_value` is the maximum of a noisy score over every setting tried.**
Take a few hundred settings of anything, score them on one stretch of
history, and the best of them looks good whether or not any of them is. That
is not a flaw in the search, it is what searching does, and it gets worse the
more trials you give it.

`oos_stats` is that same setting on bars no trial ever saw. It is the number
to quote, and it is the only one that answers the question you actually asked.

`compare` prints them side by side with a **data hash** on each row, which is
what tells you they are not the same bars. Two rows with the same hash would be
the same test twice.

`compare` prints and also returns. Naming its rows is what stops a
notebook from dumping the whole list under the table it just formatted.

The honest reading of a large gap is not that the search failed. It is that the
space contains no setting that generalises, and the search did its job by
finding the best of a bad set. **A rule with an edge shows a smaller gap, not a
bigger winner.**

In [5]:
print(f"{'in sample sharpe':<24} {study.best_value:>10,.3f}")
print(f"{'out of sample sharpe':<24} {study.oos_stats['sharpe']:>10,.3f}")
print()

rows = emsl.metrics.compare(results={
    "in sample": study.best_result,
    "out of sample": study.oos_result,
})

in sample sharpe              0.450
out of sample sharpe         -3.428

                 data      total return %          sharpe  max drawdown %      num trades
  in sample      149abcdc           6.408           0.450          24.741              25
  out of sample  acd29a83         -19.829          -3.428          27.755              15


In [6]:
#  A trial that failed min_trades scores None rather than zero.
scored = sorted((trial["value"] for trial in study.trials
                 if trial["value"] is not None), reverse=True)

print(f"{len(scored)} trials scored")
print(f"best      {scored[0]:>8,.3f}")
print(f"median    {scored[len(scored) // 2]:>8,.3f}")
print(f"worst     {scored[-1]:>8,.3f}")

200 trials scored
best         0.450
median       0.174
worst       -1.649


#### **The winner on bars it never saw**

`best_strategy()` builds a fresh strategy from the winning parameters, so the
holdout run below is the same rule the search chose, on the same bars
`oos_stats` already reported. It is re-run here only because a chart needs the
strategy's own arrays, and a finished result does not carry them.

**The order of those two lines matters.** A fresh strategy has its parameters
and nothing else, because `init` is what fills `self.fast`. Calling
`winner.marks()` before the run raises an `AttributeError` rather than drawing
an empty chart, which is the right failure but an easy one to be surprised by.

The holdout has to be the bars `oos=0.3` held back, so the split is seventy
percent of the series rather than a round number of your own.

Read the equity panel, not the return. A holdout that ends near where it
started can still be a rule that was in and out at the wrong moments the whole
way, and the drawdown panel is where that shows.

In [7]:
split   = int(len(candles) * 0.7)
holdout = candles.iloc[split:]
winner  = study.best_strategy()

on_holdout = VENUE.backtest(candles=holdout).run(strategy=winner)

print(f"holdout {len(holdout)} bars, "
      f"{holdout.index[0]} to {holdout.index[-1]}")

emsl.chart(
    frame=holdout,
    marks=winner.marks(),
    run=on_holdout,
    panels=[
        emsl.plot.Panel(name="volume", show=False),
        emsl.plot.Panel(name="equity", weight=2.0),
    ],
    title=f"winner {study.best_params} on bars no trial saw",
).show()

holdout 2628 bars, 2025-09-13 13:00:00+00:00 to 2026-01-01 00:00:00+00:00


#### **Refitting as it goes**

One holdout is one test. Score enough candidates against the same held-back
bars and you have started fitting those too, more slowly and less visibly.

`walk_forward` refits instead. `train` is the share of the series the first fit
gets, what remains is divided into `windows` stretches, and each stretch is
traded with parameters chosen on the bars before it and never on itself. It is
one continuous account rather than five, so the equity curve carries across the
seams, and it is flat until the first window because before that there was
nothing fitted to trade.

**`decay` is the mean gap between what a fit scored and what those bars then
earned**, in the units of the objective. `consistency` is the share of windows
that finished above zero, blunt on purpose: clearing it in one window out of
five is one good quarter rather than an edge.

Read the fitted column against the traded one. Two of these windows were
fitted to nearly the same score and then did nothing like the same thing,
which is the whole of it: the fitted number carries no information about the
stretch that follows it.

The chart shades each window and carries that table under the plot, so a
file saved from this cell holds the parameters every stretch was traded
on. Without it a reader has the claim and none of the evidence for it.

**The layout is a hyperparameter too.** Trying five window counts and reporting
the best of them is the same mistake one level up, and nothing here can catch
you doing it.

In [8]:
walk = VENUE.walk_forward(
    strategy=EmaCross,
    space=SPACE,
    data=candles,
    windows=5,
    train=0.5,
    objective="sharpe",
    n_trials=40,
    n_jobs=1,
    seed=7,
    min_trades=10,
)

print(f"{walk}\n")
print(f"decay        {walk.decay:>7,.3f}   sharpe the fits gave back")
print(f"consistency  {walk.consistency:>7,.0%}   of windows finished above zero\n")
print(f"  {'traded on':<16} {'chosen':<26} {'fitted':>8} {'traded':>9}")

#  One pass builds all three: the rows printed here, the shading that marks where
#  each window traded, and the table the chart carries. Walking walk.windows three
#  times would let the picture and the rows beneath it drift apart.
COOL = ["rgba(77,159,255,0.16)", "rgba(77,159,255,0.01)"]
WARM = ["rgba(139,151,165,0.18)", "rgba(139,151,165,0.01)"]

window_of = [None] * len(candles)
shading   = {}
rows      = [["window", "traded on", "chosen", "fitted", "traded"]]

for number, window in enumerate(walk.windows, start=1):
    label      = f"window {number}"
    start, end = window["traded_on"]
    traded     = window["traded"]
    shown      = "n/a" if traded is None else f"{traded:,.3f}"

    for bar in range(start, end):
        window_of[bar] = label

    shading[label] = COOL if number % 2 else WARM
    rows.append([
        number,
        f"{candles.index[start]:%Y-%m-%d} to {candles.index[end - 1]:%Y-%m-%d}",
        str(window["params"]),
        f"{window['fitted']:,.3f}",
        shown,
    ])

    print(f"  {str(window['traded_on']):<16} {str(window['params']):<26} "
          f"{window['fitted']:>8,.3f} {shown:>9}")

emsl.chart(
    frame=candles,
    run=walk.result,
    marks=emsl.plot.Background(values=window_of, name="walk", fill=shading),
    panels=[
        emsl.plot.Panel(name="volume", show=False),
        emsl.plot.Panel(name="equity", weight=2.0),
    ],
    title="each stretch traded on parameters fitted only on the bars before it",
    notes=rows,
).show()

WalkForward(5 windows, sharpe=-2.303 out of sample)

decay          4.150   sharpe the fits gave back
consistency      20%   of windows finished above zero

  traded on        chosen                       fitted    traded
  (4380, 5256)     {'fast': 20, 'slow': 199}     0.278    -3.462
  (5256, 6132)     {'fast': 17, 'slow': 200}     0.476    -0.349
  (6132, 7008)     {'fast': 38, 'slow': 198}     1.873     0.645
  (7008, 7884)     {'fast': 20, 'slow': 167}     1.840    -5.707
  (7884, 8760)     {'fast': 16, 'slow': 182}     0.286    -7.125


#### **Closing**

**Never quote the first number.** A search reports the best of everything it
tried on bars it could see, and that figure rises with the number of trials
whether or not anything in the space works. Quoting it is not optimism, it is a
measurement of how hard you looked.

**Hold something back, always.** Without `oos` there is no second number and no
way to tell a rule from a coincidence. emsl says so by leaving `oos_stats` as
`None` rather than filling it with something reassuring.

**Read `decay` before the return.** One search reports one number; refitting
reports how much its fits kept giving back, which is the same question the
holdout asked, asked repeatedly. What is still not here is
`emsl.metrics.deflated_sharpe`, which asks whether a winner beats what the
best of that many looks would have reached by luck alone. It requires a
random-search null and refuses to invent one.

`client.close()` shuts down the background thread the client runs its async loop
on, and its connection pool.

In [9]:
client.close()